# 🐍 MAMBA From Scratch — Complete Tutorial & Demo
### Deep Learning Project · ENIS 2025–2026

> **Architecture:** MAMBA (Selective State Space Model) in PyTorch from scratch  
> **Task:** Character-level Language Modeling on TinyShakespeare  
> **Key feature:** Parallel Associative Scan — O(log L) depth

---

## Table of Contents
1. Setup & Imports
2. Dataset Preparation
3. SSM Theory & Math
4. Parallel Associative Scan
5. MAMBA Architecture
6. Transformer Baseline
7. Training Both Models
8. Results & Visualizations
9. Text Generation Demo
10. Inference Speed Benchmark
11. Complexity Analysis
12. Summary & Conclusions

---
### Tools Used (AI-Powered)
| Tool | Usage |
|------|-------|
| **Claude (Anthropic)** | Code generation, explanations, architecture design |
| **GitHub Copilot** | Code completion |
| **ChatGPT** | Cross-checking mathematical derivations |
| **Gamma.app** | Presentation generation |
| **DALL-E / Midjourney** | AI-generated images for slides |


## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass
import time
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  No GPU detected — enable GPU in Runtime > Change runtime type.')

torch.manual_seed(42)
np.random.seed(42)
print('Setup complete')

## 2. Dataset — TinyShakespeare

Character-level tokenisation, 90/10 train/val split.

In [ ]:
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'shakespeare.txt')
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

chars      = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

data       = torch.tensor(encode(text), dtype=torch.long)
n          = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]

print(f'Total chars : {len(text):,}')
print(f'Vocab size  : {vocab_size}')
print(f'Train tokens: {len(train_data):,}')
print(f'Val tokens  : {len(val_data):,}')
print(f'Sample      :\n{text[:200]}')

def get_batch(split, batch_size, block_size):
    d  = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x  = torch.stack([d[i : i+block_size]   for i in ix])
    y  = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

## 3. SSM Theory & Mathematical Foundations

### State Space Model (SSM)

A continuous-time SSM:

$$h'(t) = A\,h(t) + B\,u(t)$$
$$y(t)  = C\,h(t) + D\,u(t)$$

Discretised via Zero-Order Hold (step $\Delta$):

$$\bar{A} = e^{\Delta A}, \quad \bar{B} \approx \Delta B$$
$$h_t = \bar{A}\,h_{t-1} + \bar{B}\,u_t, \quad y_t = C\,h_t + D\,u_t$$

### Why MAMBA is "Selective" (S6)

Classical SSMs (S4): A, B, C are **fixed** for every input.  
MAMBA: **B, C, and Δ are functions of the current token** $u_t$:

$$\Delta_t,\; B_t,\; C_t = f(u_t)$$

This lets the model selectively remember or forget — like attention, but O(L log L) not O(L²).

| Model | Parameters | Complexity | Memory |
|-------|-----------|------------|--------|
| S4 | Fixed A,B,C | O(L) | O(N) |
| **MAMBA (S6)** | **Input-dep.** | **O(L log L)** | **O(L)** |
| Transformer | — | O(L²) | O(L²) |


## 4. Parallel Associative Scan

The recurrence $h_t = \bar{A}_t h_{t-1} + \bar{B}_t u_t$ looks sequential.
A naive loop = O(L) sequential steps. **No GPU parallelism.**

The operator:
$$(g_2, v_2) \oplus (g_1, v_1) = (g_2 \cdot g_1,\; g_2 \cdot v_1 + v_2)$$

is **associative** → Blelloch divide-and-conquer:

```
L=8:  h0  h1  h2  h3  h4  h5  h6  h7
Level1: h01  h23  h45  h67       (4 parallel ops)
Level2:  h0123    h4567           (2 parallel ops)
Level3:   h01234567               (1 op)
=> Only log2(L) = 3 passes for L=8
```


In [ ]:
def parallel_scan(gates, tokens):
    """
    Parallel prefix scan: h_t = gate_t * h_{t-1} + token_t
    Complexity: O(L log L) work, O(log L) depth (vs O(L) sequential).

    Args:
        gates  : (B, L, D, N)
        tokens : (B, L, D, N)
    Returns:
        h      : (B, L, D, N)
    """
    B, L, D, N = gates.shape
    if L == 1:
        return tokens
    if L % 2 == 1:
        gates  = F.pad(gates,  (0, 0, 0, 0, 0, 1))
        tokens = F.pad(tokens, (0, 0, 0, 0, 0, 1))
    Lp = gates.shape[1]

    g_even, g_odd = gates[:, 0::2], gates[:, 1::2]
    t_even, t_odd = tokens[:, 0::2], tokens[:, 1::2]

    # Associative combine: (g2, v2) o (g1, v1) = (g2*g1, g2*v1+v2)
    g_new = g_odd * g_even
    t_new = g_odd * t_even + t_odd

    h_odd  = parallel_scan(g_new, t_new)
    h_prev = F.pad(h_odd[:, :-1], (0, 0, 0, 0, 1, 0))
    h_even = g_even * h_prev + t_even

    h = torch.stack([h_even, h_odd], dim=2).reshape(B, Lp, D, N)
    return h[:, :L]


def sequential_scan(gates, tokens):
    """Reference sequential scan for verification."""
    B, L, D, N = gates.shape
    h = torch.zeros(B, D, N, device=gates.device)
    hs = []
    for t in range(L):
        h = gates[:, t] * h + tokens[:, t]
        hs.append(h)
    return torch.stack(hs, dim=1)


# Correctness test
torch.manual_seed(0)
g_ = torch.sigmoid(torch.randn(2, 16, 8, 4))
t_ = torch.randn(2, 16, 8, 4)
err = (sequential_scan(g_, t_) - parallel_scan(g_, t_)).abs().max().item()
print(f'Correctness check: max error = {err:.2e}  =>  {"PASSED" if err < 1e-5 else "FAILED"}')

# Speed benchmark
print()
print(f'  {"L":<8} {"Sequential (ms)":<22} {"Parallel (ms)":<22} Speedup')
for l in [64, 128, 256, 512, 1024]:
    g_ = torch.sigmoid(torch.randn(4, l, 16, 8))
    t_ = torch.randn(4, l, 16, 8)
    reps = 30
    t0 = time.time()
    for _ in range(reps): sequential_scan(g_, t_)
    ts = (time.time()-t0)/reps*1000
    t0 = time.time()
    for _ in range(reps): parallel_scan(g_, t_)
    tp = (time.time()-t0)/reps*1000
    print(f'  {l:<8} {ts:<22.2f} {tp:<22.2f} {ts/tp:.2f}x')

## 5. MAMBA Architecture

```
Input tokens
     |
Embedding (vocab -> d_model)
     |  x N_LAYERS
+------------------------------+
|       MambaBlock             |
|  LayerNorm + residual        |
|  in_proj  ->  x | z (gate)  |
|  Conv1D (local context)      |
|  SiLU                        |
|  SelectiveSSM (S6)           |
|    A (fixed log-param)       |
|    B, C, Delta (input-dep.)  |
|    Parallel Scan             |
|  x SiLU(z)  + out_proj      |
+------------------------------+
     |
LayerNorm -> LM Head -> Logits
```


In [ ]:
@dataclass
class MambaConfig:
    d_model  : int   = 128
    d_state  : int   = 16
    d_conv   : int   = 4
    expand   : int   = 2
    dt_rank  : str   = 'auto'
    dt_min   : float = 0.001
    dt_max   : float = 0.1
    bias     : bool  = False
    conv_bias: bool  = True

    def __post_init__(self):
        self.d_inner = int(self.expand * self.d_model)
        if self.dt_rank == 'auto':
            self.dt_rank = math.ceil(self.d_model / 16)


class SelectiveSSM(nn.Module):
    """
    S6 — Selective State Space Model.
    B, C, Delta are INPUT-DEPENDENT (key MAMBA innovation).
    Uses parallel_scan for O(log L) sequential depth.
    """
    def __init__(self, cfg: MambaConfig):
        super().__init__()
        self.d_inner = cfg.d_inner
        self.d_state = cfg.d_state
        self.dt_rank = cfg.dt_rank

        A = torch.arange(1, cfg.d_state+1, dtype=torch.float32).repeat(cfg.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))   # log-param for stability
        self.D     = nn.Parameter(torch.ones(cfg.d_inner))

        # Projects input to [dt_low | B_sel | C_sel]
        self.x_proj  = nn.Linear(cfg.d_inner, cfg.dt_rank + cfg.d_state*2, bias=False)
        # Low-rank Delta -> full d_inner
        self.dt_proj = nn.Linear(cfg.dt_rank, cfg.d_inner, bias=True)

        # Init Delta log-uniformly in [dt_min, dt_max] (from paper)
        nn.init.uniform_(self.dt_proj.weight, -(cfg.dt_rank**-0.5), cfg.dt_rank**-0.5)
        dt = torch.exp(
            torch.rand(cfg.d_inner) * (math.log(cfg.dt_max) - math.log(cfg.dt_min))
            + math.log(cfg.dt_min)
        ).clamp(min=1e-4)
        with torch.no_grad():
            self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))

    def forward(self, x):
        """x: (B, L, d_inner) -> y: (B, L, d_inner)"""
        B, L, d = x.shape
        A = -torch.exp(self.A_log.float())   # always negative

        x_dbl            = self.x_proj(x)
        dt, B_sel, C_sel = x_dbl.split([self.dt_rank, self.d_state, self.d_state], dim=-1)
        dt = F.softplus(self.dt_proj(dt))    # positive step size (B, L, d_inner)

        # ZOH discretisation
        dA  = torch.exp(torch.einsum('bld,dn->bldn', dt, A))
        dBu = torch.einsum('bld,bln,bld->bldn', dt, B_sel, x)

        # PARALLEL SCAN instead of sequential for-loop
        h = parallel_scan(dA, dBu)           # (B, L, d_inner, d_state)

        y = torch.einsum('bldn,bln->bld', h, C_sel)
        y = y + x * self.D                   # D skip connection
        return y


class MambaBlock(nn.Module):
    """
    MAMBA block:
    LayerNorm -> in_proj(x2) -> Conv1D -> SiLU -> SelectiveSSM -> gate -> out_proj -> residual
    """
    def __init__(self, cfg: MambaConfig):
        super().__init__()
        d, di = cfg.d_model, cfg.d_inner
        self.norm     = nn.LayerNorm(d)
        self.in_proj  = nn.Linear(d, di*2, bias=cfg.bias)
        self.conv1d   = nn.Conv1d(di, di, kernel_size=cfg.d_conv,
                                  padding=cfg.d_conv-1, groups=di, bias=cfg.conv_bias)
        self.ssm      = SelectiveSSM(cfg)
        self.out_proj = nn.Linear(di, d, bias=cfg.bias)

    def forward(self, x):
        residual  = x
        x         = self.norm(x)
        x_main, z = self.in_proj(x).chunk(2, dim=-1)

        # Causal depthwise conv for local context
        x_conv = self.conv1d(x_main.transpose(1,2))[:, :, :x_main.size(1)]
        x_conv = F.silu(x_conv.transpose(1,2))

        y = self.ssm(x_conv)
        y = y * F.silu(z)                    # gating
        return self.out_proj(y) + residual


class MambaLM(nn.Module):
    """Full MAMBA language model: Embedding -> N x MambaBlock -> LM Head."""
    def __init__(self, vocab_size, n_layers, cfg):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, cfg.d_model)
        self.layers    = nn.ModuleList([MambaBlock(cfg) for _ in range(n_layers)])
        self.norm      = nn.LayerNorm(cfg.d_model)
        self.lm_head   = nn.Linear(cfg.d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embedding.weight   # weight tying

    def forward(self, idx, targets=None):
        x      = self.embedding(idx)
        for layer in self.layers:
            x  = layer(x)
        logits = self.lm_head(self.norm(x))
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, block_size=128):
        """Autoregressive generation with top-k sampling."""
        for _ in range(max_new_tokens):
            idx_cond  = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx   = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

print('MAMBA architecture defined: MambaConfig | SelectiveSSM | MambaBlock | MambaLM')

## 6. Transformer Baseline

Mini GPT (same `d_model=128`, 4 layers) for fair comparison.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, 4*d_model), nn.GELU(),
            nn.Linear(4*d_model, d_model), nn.Dropout(dropout),
        )
        self.register_buffer('mask',
            torch.triu(torch.ones(block_size, block_size), diagonal=1).bool())

    def forward(self, x):
        L    = x.size(1)
        mask = self.mask[:L, :L]
        a, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x),
                         attn_mask=mask, is_causal=False)
        x = x + a
        x = x + self.ffn(self.norm2(x))
        return x


class TransformerLM(nn.Module):
    def __init__(self, vocab_size, n_layers, d_model, n_heads, block_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.blocks  = nn.ModuleList([
            TransformerBlock(d_model, n_heads, block_size) for _ in range(n_layers)
        ])
        self.norm    = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None):
        B, L   = idx.shape
        x      = self.tok_emb(idx) + self.pos_emb(torch.arange(L, device=idx.device))
        for block in self.blocks:
            x  = block(x)
        logits = self.lm_head(self.norm(x))
        loss   = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, block_size=128):
        for _ in range(max_new_tokens):
            idx_cond  = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits    = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx   = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx

print('Transformer baseline defined: TransformerBlock | TransformerLM')

## 7. Training Both Models

In [ ]:
BATCH_SIZE = 32
BLOCK_SIZE = 128
MAX_ITERS  = 5000
EVAL_ITERS = 100
EVAL_EVERY = 500
LR         = 3e-4
N_LAYERS   = 4
N_HEADS    = 4

cfg = MambaConfig(d_model=128, d_state=16, d_conv=4, expand=2)

# FIX: count_params defined here before first use
def count_params(m):
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

mamba_model = MambaLM(vocab_size=vocab_size, n_layers=N_LAYERS, cfg=cfg).to(device)
tfm_model   = TransformerLM(vocab_size=vocab_size, n_layers=N_LAYERS,
                              d_model=cfg.d_model, n_heads=N_HEADS,
                              block_size=BLOCK_SIZE).to(device)

print(f'MAMBA       : {count_params(mamba_model):>10,} parameters')
print(f'Transformer : {count_params(tfm_model):>10,} parameters')
print(f'Ratio       : {count_params(mamba_model)/count_params(tfm_model):.2f}x')

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = [model(*get_batch(split, BATCH_SIZE, BLOCK_SIZE))[1].item()
                  for _ in range(EVAL_ITERS)]
        out[split] = np.mean(losses)
    model.train()
    return out


def train_model(model, name, max_iters=MAX_ITERS):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max_iters, eta_min=LR/10)
    train_losses, val_losses, iter_log, step_times = [], [], [], []
    t0 = time.time()

    for it in range(max_iters):
        if it % EVAL_EVERY == 0 or it == max_iters - 1:
            losses  = estimate_loss(model)
            elapsed = time.time() - t0
            print(f'[{name}] iter {it:5d}/{max_iters}  '
                  f'train={losses["train"]:.4f}  val={losses["val"]:.4f}  '
                  f'ppl={math.exp(losses["val"]):.1f}  ({elapsed:.0f}s)')
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])
            iter_log.append(it)

        t_step = time.time()
        xb, yb = get_batch('train', BATCH_SIZE, BLOCK_SIZE)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        step_times.append(time.time() - t_step)

    total_time = time.time() - t0
    print(f'\n[{name}] Done in {total_time:.0f}s  |  '
          f'Best val loss: {min(val_losses):.4f}  |  '
          f'Best ppl: {math.exp(min(val_losses)):.1f}')
    return {
        'train_losses': train_losses, 'val_losses': val_losses,
        'iter_log': iter_log, 'total_time': total_time,
        'avg_step_ms': np.mean(step_times) * 1000,
    }

print(f'Training functions ready — {MAX_ITERS} iters, eval every {EVAL_EVERY}')

In [ ]:
print('='*65)
print('TRAINING MAMBA')
print('='*65)
mamba_hist = train_model(mamba_model, 'MAMBA')

In [ ]:
print('='*65)
print('TRAINING TRANSFORMER')
print('='*65)
tfm_hist = train_model(tfm_model, 'Transformer')

## 8. Results & Visualizations

In [ ]:
BG_DARK  = '#0D1B2A'
BG_PANEL = '#152535'
SPINE    = '#1E3A5F'
TXT      = '#94A3B8'
C_MAMBA  = '#06B6D4'
C_TFM    = '#F59E0B'

def style_ax(ax):
    ax.set_facecolor(BG_PANEL)
    ax.tick_params(colors=TXT, labelsize=9)
    for sp in ax.spines.values():
        sp.set_color(SPINE); sp.set_linewidth(0.8)
    ax.title.set_color('white')
    ax.xaxis.label.set_color(TXT)
    ax.yaxis.label.set_color(TXT)

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor(BG_DARK)
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
iter_log = mamba_hist['iter_log']

ax1 = fig.add_subplot(gs[0, 0]); style_ax(ax1)
ax1.plot(iter_log, mamba_hist['val_losses'], color=C_MAMBA, lw=2.5, marker='o', ms=4, label='MAMBA')
ax1.plot(iter_log, tfm_hist['val_losses'],   color=C_TFM,   lw=2.5, marker='s', ms=4, ls='--', label='Transformer')
ax1.set_title('Validation Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Cross-Entropy Loss')
ax1.legend(facecolor=BG_DARK, edgecolor=SPINE, labelcolor='white', fontsize=9)
ax1.grid(True, alpha=0.2)

ax2 = fig.add_subplot(gs[0, 1]); style_ax(ax2)
ax2.plot(iter_log, [math.exp(l) for l in mamba_hist['val_losses']], color=C_MAMBA, lw=2.5, marker='o', ms=4, label='MAMBA')
ax2.plot(iter_log, [math.exp(l) for l in tfm_hist['val_losses']],   color=C_TFM,   lw=2.5, marker='s', ms=4, ls='--', label='Transformer')
ax2.set_title('Validation Perplexity', fontsize=13, fontweight='bold')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Perplexity')
ax2.legend(facecolor=BG_DARK, edgecolor=SPINE, labelcolor='white', fontsize=9)
ax2.grid(True, alpha=0.2)

ax3 = fig.add_subplot(gs[0, 2]); style_ax(ax3)
ax3.plot(iter_log, mamba_hist['train_losses'], color=C_MAMBA,   lw=2.5, marker='o', ms=4, label='Train')
ax3.plot(iter_log, mamba_hist['val_losses'],   color='#7C3AED', lw=2.5, marker='s', ms=4, ls='--', label='Val')
ax3.set_title('MAMBA: Train vs Validation', fontsize=13, fontweight='bold')
ax3.set_xlabel('Iteration'); ax3.set_ylabel('Loss')
ax3.legend(facecolor=BG_DARK, edgecolor=SPINE, labelcolor='white', fontsize=9)
ax3.grid(True, alpha=0.2)

ax4 = fig.add_subplot(gs[1, 0]); style_ax(ax4)
methods   = ['LSTM\n(sequential)', 'Transformer\n(attention)', 'MAMBA\n(parallel scan)']
seq_steps = [128, 128*128, math.log2(128)]
colors_b  = ['#EF4444', C_TFM, C_MAMBA]
bars = ax4.bar(methods, seq_steps, color=colors_b, width=0.5, edgecolor='white', linewidth=0.5)
ax4.set_title('Sequential Steps (L=128)', fontsize=13, fontweight='bold')
ax4.set_ylabel('# Sequential Steps (log scale)')
ax4.set_yscale('log')
for bar, val in zip(bars, seq_steps):
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.1,
             f'{val:.0f}' if val>=1 else f'{val:.1f}',
             ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
ax4.grid(True, alpha=0.2, axis='y')

ax5 = fig.add_subplot(gs[1, 1]); style_ax(ax5)
step_t = [mamba_hist['avg_step_ms'], tfm_hist['avg_step_ms']]
bars2  = ax5.bar(['MAMBA', 'Transformer'], step_t,
                 color=[C_MAMBA, C_TFM], width=0.4, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars2, step_t):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
             f'{val:.1f}ms', ha='center', va='bottom', color='white', fontsize=11, fontweight='bold')
ax5.set_title('Avg Step Time (ms)', fontsize=13, fontweight='bold')
ax5.set_ylabel('Milliseconds per step')
ax5.grid(True, alpha=0.2, axis='y')

ax6 = fig.add_subplot(gs[1, 2]); style_ax(ax6)
metrics    = ['Best\nVal Loss', 'Best\nPerplexity', 'Params (K)']
mamba_vals = [min(mamba_hist['val_losses']),
              math.exp(min(mamba_hist['val_losses'])),
              count_params(mamba_model)/1000]
tfm_vals   = [min(tfm_hist['val_losses']),
              math.exp(min(tfm_hist['val_losses'])),
              count_params(tfm_model)/1000]
x = np.arange(len(metrics)); w = 0.3
ax6.bar(x-w/2, mamba_vals, w, color=C_MAMBA, label='MAMBA',       edgecolor='white', lw=0.5)
ax6.bar(x+w/2, tfm_vals,   w, color=C_TFM,   label='Transformer', edgecolor='white', lw=0.5)
ax6.set_xticks(x); ax6.set_xticklabels(metrics, fontsize=9)
ax6.legend(facecolor=BG_DARK, edgecolor=SPINE, labelcolor='white', fontsize=9)
ax6.set_title('Final Comparison', fontsize=13, fontweight='bold')
ax6.grid(True, alpha=0.2, axis='y')

fig.suptitle('MAMBA vs Transformer — Character-Level Language Modeling',
             color='white', fontsize=16, fontweight='bold', y=1.01)
plt.savefig('full_results.png', dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()
print('Saved: full_results.png')

## 9. Text Generation Demo

Same prompts fed to both models at varying temperatures.

In [ ]:
def generate_text(model, prompt, max_new_tokens=300, temperature=0.8, top_k=40):
    model.eval()
    ctx = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(ctx, max_new_tokens, temperature=temperature,
                         top_k=top_k, block_size=BLOCK_SIZE)
    return decode(out[0].tolist())

prompts = [
    ('HAMLET:\n',           0.8, 40),
    ('To be or not to be',   0.7, 40),
    ('KING:\nSpeak, villain',0.6, 40),
]

for prompt, temp, k in prompts:
    print('\n' + '='*65)
    print(f'Prompt: {repr(prompt)}  |  temperature={temp}  top_k={k}')
    print('-'*65)
    print('MAMBA output:')
    print(generate_text(mamba_model, prompt, temperature=temp, top_k=k))
    print('-'*65)
    print('Transformer output:')
    print(generate_text(tfm_model, prompt, temperature=temp, top_k=k))

## 10. Inference Speed Benchmark

Generation throughput (tokens/second) at sequence length 128.

In [ ]:
print('Inference speed benchmark (tokens/second)')
print('-'*50)

n_tokens  = 500
n_repeats = 3
prompt_t  = torch.zeros(1, 1, dtype=torch.long, device=device)

def bench_model(model, name):
    times = []
    for _ in range(n_repeats):
        t0 = time.time()
        with torch.no_grad():
            model.generate(prompt_t, max_new_tokens=n_tokens,
                           temperature=0.8, top_k=40, block_size=BLOCK_SIZE)
        times.append(time.time() - t0)
    avg = np.mean(times)
    tps = n_tokens / avg
    print(f'  {name:<15}: {tps:>7.1f} tok/s  ({avg*1000:.0f} ms / {n_tokens} tokens)')
    return tps

mamba_tps = bench_model(mamba_model, 'MAMBA')
tfm_tps   = bench_model(tfm_model,   'Transformer')
print(f'\n  Ratio: MAMBA is {mamba_tps/tfm_tps:.2f}x faster at inference (block_size={BLOCK_SIZE})')

fig, ax = plt.subplots(figsize=(7, 4))
fig.patch.set_facecolor(BG_DARK); ax.set_facecolor(BG_PANEL)
bars = ax.bar(['MAMBA', 'Transformer'], [mamba_tps, tfm_tps],
              color=[C_MAMBA, C_TFM], width=0.4, edgecolor='white', lw=0.5)
for bar, val in zip(bars, [mamba_tps, tfm_tps]):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
            f'{val:.0f} tok/s', ha='center', va='bottom', color='white',
            fontsize=12, fontweight='bold')
ax.set_title('Inference Throughput (tok/s)', color='white', fontsize=13, fontweight='bold')
ax.set_ylabel('Tokens / second', color=TXT)
ax.tick_params(colors=TXT)
for sp in ax.spines.values(): sp.set_color(SPINE)
ax.grid(True, alpha=0.2, axis='y')
fig.tight_layout()
plt.savefig('inference_speed.png', dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()
print('Saved: inference_speed.png')

## 11. Complexity Analysis

| Model | Time | Memory | Parallelizable |
|-------|------|--------|----------------|
| RNN/LSTM | O(L) | O(1) | No |
| Transformer | O(L²) | O(L²) | Yes |
| **MAMBA (S6)** | **O(L log L)** | **O(L)** | **Yes** |

For L=4096: Transformer -> 16.7M ops · MAMBA -> 49K ops (**×341 faster**).

In [ ]:
Ls = np.array([64, 128, 256, 512, 1024, 2048, 4096, 8192])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(BG_DARK)
for ax in [ax1, ax2]:
    style_ax(ax); ax.grid(True, alpha=0.2)

ax1.plot(Ls, Ls,             color='#10B981', lw=2.5, marker='o', ms=4, label='LSTM: O(L)')
ax1.plot(Ls, Ls**2/1000,     color=C_TFM,    lw=2.5, marker='s', ms=4, label='Transformer: O(L^2) /1000')
ax1.plot(Ls, Ls*np.log2(Ls), color=C_MAMBA,  lw=2.5, marker='^', ms=4, label='MAMBA: O(L log L)')
ax1.set_title('Time Complexity Scaling', color='white', fontsize=13, fontweight='bold')
ax1.set_xlabel('Sequence Length L'); ax1.set_ylabel('Operations (relative)')
ax1.legend(facecolor=BG_DARK, edgecolor=SPINE, labelcolor='white', fontsize=9)
ax1.set_yscale('log'); ax1.set_xscale('log')

ax2.plot(Ls, np.ones_like(Ls), color='#10B981', lw=2.5, marker='o', ms=4, label='LSTM: O(1)')
ax2.plot(Ls, Ls**2/10000,      color=C_TFM,    lw=2.5, marker='s', ms=4, label='Transformer: O(L^2) /10000')
ax2.plot(Ls, Ls/100,           color=C_MAMBA,  lw=2.5, marker='^', ms=4, label='MAMBA: O(L) /100')
ax2.set_title('Memory Complexity Scaling', color='white', fontsize=13, fontweight='bold')
ax2.set_xlabel('Sequence Length L'); ax2.set_ylabel('Memory (relative)')
ax2.legend(facecolor=BG_DARK, edgecolor=SPINE, labelcolor='white', fontsize=9)
ax2.set_yscale('log'); ax2.set_xscale('log')

fig.suptitle('MAMBA vs Transformer vs LSTM — Scaling Laws',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('complexity_analysis.png', dpi=150, bbox_inches='tight', facecolor=BG_DARK)
plt.show()
print('Saved: complexity_analysis.png')

## 12. Summary & Conclusions

### What we built
- MAMBA from scratch in PyTorch: Selective SSM (S6) + Parallel Associative Scan
- Transformer baseline for fair comparison
- Both trained on TinyShakespeare character-level language modeling

### Key findings
1. MAMBA achieves comparable perplexity to Transformer with fewer parameters
2. MAMBA scales better: O(L log L) vs O(L²)
3. Selective state spaces allow content-based filtering like attention, but efficiently
4. Parallel scan is the engineering key: O(log L) depth on GPU

### MAMBA real-world applications
- Medical imaging (MRI/CT segmentation)
- Remote sensing (satellite imagery)
- Motion generation from text
- Long time-series prediction

### References
- Gu & Dao (2023). Mamba: Linear-Time Sequence Modeling with Selective State Spaces. arXiv:2312.00752
- Gu et al. (2021). S4: Structured State Spaces. arXiv:2111.00396


In [ ]:
torch.save({
    'mamba_state': mamba_model.state_dict(),
    'tfm_state':   tfm_model.state_dict(),
    'mamba_hist':  mamba_hist,
    'tfm_hist':    tfm_hist,
    'cfg':         cfg,
    'vocab_size':  vocab_size,
    'stoi':        stoi,
    'itos':        itos,
}, 'mamba_final.pt')
print('Saved: mamba_final.pt')

print()
print('='*65)
print('                 FINAL RESULTS SUMMARY')
print('='*65)
print(f'  {"Metric":<28} {"MAMBA":>14} {"Transformer":>14}')
print('-'*65)
print(f'  {"Best Val Loss":<28} {min(mamba_hist["val_losses"]):>14.4f} {min(tfm_hist["val_losses"]):>14.4f}')
print(f'  {"Best Perplexity":<28} {math.exp(min(mamba_hist["val_losses"])):>14.1f} {math.exp(min(tfm_hist["val_losses"])):>14.1f}')
print(f'  {"Parameters":<28} {count_params(mamba_model):>14,} {count_params(tfm_model):>14,}')
print(f'  {"Avg Step Time (ms)":<28} {mamba_hist["avg_step_ms"]:>14.1f} {tfm_hist["avg_step_ms"]:>14.1f}')
print(f'  {"Total Train Time (s)":<28} {mamba_hist["total_time"]:>14.0f} {tfm_hist["total_time"]:>14.0f}')
print('='*65)
print()
print('MAMBA implementation complete!')